# Place Each Statement under a Codebook Topic

| Item | Description |
|---|---|
| Goal | Place each of the 32,385 stance records of T2.2.4 under the nearest topic of the frozen codebook. The record's topic phrase is embedded and compared with each topic's centroid, the weighted mean embedding of the topic's Track 1 phrases from T2.1.4. A record is placed when the best cosine is at least 0.68; below that it stays in the explicit other pool as fine-grained detail. |
| Input | `stance_records.parquet` from T2.2.4 in the T2.2 Processed folder; `codebook_v1.parquet` and `topic_centroids.npz` from T2.1.4; the encoder BAAI/bge-large-en-v1.5 from the project cache. |
| Output | `track2_assigned.parquet` in the T2.2 Processed folder: the stance records with `topic_id`, `topic_label`, and `sim`. T2.4, T2.5, and T3 read it. |
| Variables | • `sim`: the cosine between the record's phrase and its nearest topic centroid<br>• `topic_id`: the placed topic, or -1 for the other pool<br>• `TAU`: the floor of 0.68, set from a reading of the phrases just above and below it in the earlier run |
| Run | The placement ran on 2026-08-22 on one A40; embedding the 24,798 distinct phrases is its only GPU step. Not rerun: T2.4, T2.5, and T3 were computed on this placement. Step 6 reads the stored output on 2026-10-31 and reports its coverage and the phrases at the floor. |


## Step 1. Import packages


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer


## Step 2. Define file locations


In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.2 - Track 2 - Legislator Stance Extraction")
NOTEBOOK_NAME = "T2.2.6 - Assign - Place Each Statement under a Codebook Topic"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
INPUT_FOLDER = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.1 - Track 1 - Topic Extraction based on Hearing Summaries" / "Processed"
TAU = 0.68
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 3. Define the encoder call


In [ ]:
ENCODER = "BAAI/bge-large-en-v1.5"  # the encoder of T1.3.2 and T3; CLS pooling is the pooling this checkpoint publishes


@torch.no_grad()
def encode(texts, model, tokenizer, batch_size=256, max_length=512):
    """Embed texts: CLS token, L2-normalized, stored in half precision. Cosine similarity is then a dot product."""
    out = np.empty((len(texts), model.config.hidden_size), dtype=np.float16)
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(model.device)
        vec = torch.nn.functional.normalize(model(**enc).last_hidden_state[:, 0], dim=-1)
        out[start:start + len(batch)] = vec.float().cpu().numpy().astype(np.float16)
    return out


## Step 4. Load the codebook, the centroids, and the stance records


In [3]:
meta = pd.read_parquet(INPUT_FOLDER / "codebook_v1.parquet").set_index("topic_id")
cent = np.load(INPUT_FOLDER / "topic_centroids.npz")
C, tids = cent["C"].astype(np.float32), cent["topic_id"]
t2 = pd.read_parquet(PROCESSED_FOLDER / "stance_records.parquet")
t2["key"] = t2["topic"].str.casefold().str.strip()
uniq2 = t2["key"].drop_duplicates()
print(f"{len(meta)} topics, {len(t2):,} stance records, {len(uniq2):,} distinct topic phrases to embed")


111 topics, 32,385 stance records, 24,798 distinct topic phrases to embed


## Step 5. Embed the phrases and place the records

Each distinct phrase is embedded once. Its nearest centroid is the candidate topic, accepted at cosine 0.68 or above.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(ENCODER)
encoder = AutoModel.from_pretrained(ENCODER, dtype="auto", device_map=device).eval()
E2 = encode(uniq2.tolist(), encoder, tokenizer).astype(np.float32)
S = E2 @ C.T
best = S.argmax(1)
sim = S.max(1)
u2 = pd.DataFrame({"key": uniq2.to_numpy(), "sim": sim, "topic_id": np.where(sim >= TAU, tids[best], -1)})

t2 = t2.merge(u2, on="key", how="left")
t2["topic_label"] = t2["topic_id"].map(meta["label"]).fillna("other")
t2.drop(columns=["key"]).to_parquet(PROCESSED_FOLDER / "track2_assigned.parquet", index=False)
print(f"records placed (cosine >= {TAU}): {(t2['topic_id'] >= 0).mean():.1%}; median cosine of placed records {t2.loc[t2['topic_id'] >= 0, 'sim'].median():.2f}")


## Step 6. Check the stored placement

Read back from disk: coverage, the median cosine, and eight phrases drawn from the band just above the floor, with the topic they were given. The band shows what the floor lets through; a few of these placements are wrong, which is why the record keeps the original phrase.


In [4]:
back = pd.read_parquet(PROCESSED_FOLDER / "track2_assigned.parquet")
assert len(back) == 32385 and ((back["sim"] >= TAU) == (back["topic_id"] >= 0)).all()
placed = back["topic_id"] >= 0
print(f"1) records placed : {placed.mean():.1%} of {len(back):,}; other pool {(~placed).sum():,}")
print(f"2) cosine         : median {back.loc[placed, 'sim'].median():.2f} among placed records")
print(f"3) topics used    : {back.loc[placed, 'topic_id'].nunique()} of {len(meta)}")
band = (back.assign(key=back["topic"].str.casefold().str.strip()).drop_duplicates("key")
        .loc[lambda d: (d["sim"] >= TAU) & (d["sim"] < TAU + 0.04)].sample(8, random_state=1089))
print(f"\n4) phrases in the floor band ({TAU} to {TAU + 0.04:.2f}):")
for r in band.itertuples():
    print(f"   {r.sim:.2f}  {r.topic[:48]:48s} -> {r.topic_label[:44]}")


1) records placed : 92.6% of 32,385; other pool 2,406
2) cosine         : median 0.78 among placed records
3) topics used    : 111 of 111

4) phrases in the floor band (0.68 to 0.72):
   0.69  feedback to children                             -> Children's online privacy and safety legisla
   0.72  ATP and MEP zeroing out                          -> stem cell research and bioethics
   0.68  NASA Inspector General Robert Cobb's misconduct  -> inspectors general and federal accountabilit
   0.69  last paragraph of the amendment                  -> hours-of-service regulations
   0.70  IFQ decisionmaking process                       -> inspectors general and federal accountabilit
   0.69  the idea                                         -> Online advertising and consumer choice polic
   0.71  company's financial and political activities     -> Consumer credit protection and loan guarante
   0.69  software for the 737 Max 8s and Max 9s           -> Aviation safety and certification policy
